In [ ]:
!pip install -q langgraph transformers torch


In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END
from transformers import pipeline



class GraphState(TypedDict):
    user_message: str
    route: Literal["python", "general"]
    final_answer: str



hf_model = pipeline(
    task="text2text-generation",
    model="google/flan-t5-base"
)



def router(state: GraphState) -> GraphState:
    message = state["user_message"].lower()

    python_keywords = [
        "python", "code", "function", "list", "dict",
        "loop", "class", "error", "exception", "algorithm"
    ]

    for kw in python_keywords:
        if kw in message:
            return {"route": "python"}

    return {"route": "general"}



def python_agent(state: GraphState) -> GraphState:
    prompt = (
        "Answer the following Python programming question clearly.\n\n"
        f"Question: {state['user_message']}\n\n"
        "Answer:"
    )

    answer = hf_model(
        prompt,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.3
    )[0]["generated_text"]

    return {"final_answer": answer.strip()}



def general_agent(state: GraphState) -> GraphState:
    prompt = (
        "Answer the following general question clearly.\n\n"
        f"Question: {state['user_message']}\n\n"
        "Answer:"
    )

    answer = hf_model(
        prompt,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.3
    )[0]["generated_text"]

    return {"final_answer": answer.strip()}



graph = StateGraph(GraphState)

graph.add_node("router", router)
graph.add_node("python_agent", python_agent)
graph.add_node("general_agent", general_agent)

graph.set_entry_point("router")

graph.add_conditional_edges(
    "router",
    lambda state: state["route"],
    {
        "python": "python_agent",
        "general": "general_agent",
    }
)

graph.add_edge("python_agent", END)
graph.add_edge("general_agent", END)

app = graph.compile()



initial_state = {
    "user_message": "What is photosynthesis?"

}

result = app.invoke(initial_state)

print("Final Answer:\n")
print(result["final_answer"])


Device set to use cpu


Final Answer:

photosynthesis is the process of converting sunlight into light energy.
